# Clean and merge telecom data with ACS income

This notebook runs the same pipeline as the scripts `clean_telco.py`, `clean_income.py`, and `merge_income.py`, and writes `telco_clean.parquet`, `income_clean.parquet`, and `telco_with_income.parquet` under `data_processed/`.

The merge is a left join on `state_abbrev` with many customers mapped to one income row per state (`validate='many_to_one'`).

In [2]:
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "src" / "config.py").exists():
    root = root.parent
if not (root / "src" / "config.py").exists():
    raise FileNotFoundError("Project root (src/config.py) not found")
sys.path.insert(0, str(root))
print("Project root:", root)

Project root: e:\Personal Project\telecom-churn-income-integration


In [3]:
from src.clean_income import write_income_clean
from src.clean_telco import write_telco_clean
from src.merge_income import write_telco_with_income

write_telco_clean()
write_income_clean()
write_telco_with_income()
print("Done: telco_clean, income_clean, telco_with_income")

Done: telco_clean, income_clean, telco_with_income


In [4]:
import pandas as pd

from src.config import TELCO_WITH_INCOME_PATH

df = pd.read_parquet(TELCO_WITH_INCOME_PATH)
print("Shape:", df.shape)
print("Missing income share:", df["median_household_income"].isna().mean().round(4))
df[["state", "state_abbrev", "median_household_income"]].head(10)

Shape: (7043, 34)
Missing income share: 0.0


,state,state_abbrev,median_household_income
0,California,CA,96334
1,California,CA,96334
2,California,CA,96334
3,California,CA,96334
4,California,CA,96334
5,California,CA,96334
6,California,CA,96334
7,California,CA,96334
8,California,CA,96334
9,California,CA,96334


In [5]:
# Merge coverage: distinct states in telco vs income match rate
vc = df["state_abbrev"].value_counts()
print("States (abbrev) in merged data:", vc.index.tolist()[:10], "... n=", len(vc))
df["median_household_income"].describe()

States (abbrev) in merged data: ['CA'] ... n= 1


count     7043.0
mean     96334.0
std          0.0
min      96334.0
25%      96334.0
50%      96334.0
75%      96334.0
max      96334.0
Name: median_household_income, dtype: float64